# Week 2 — Defining the research design

This notebook converts the class discussion into a reproducible analysis plan. It does not fit models. Its purpose is to make features, targets, data splits, dependencies and leakage boundaries explicit before modelling begins.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

ROOT = Path.cwd()
if not (ROOT / "23_wrp" / "WRP_2023.csv").exists() and (ROOT.parent / "23_wrp" / "WRP_2023.csv").exists():
    ROOT = ROOT.parent

RAW_DATA = ROOT / "23_wrp" / "WRP_2023.csv"
DICTIONARY = ROOT / "23_wrp" / "WRP_2023_data_dictionary.csv"
ARTIFACTS = ROOT / "course_artifacts"
ARTIFACTS.mkdir(exist_ok=True)

assert RAW_DATA.exists(), f"Cannot find {RAW_DATA}"
assert DICTIONARY.exists(), f"Cannot find {DICTIONARY}"

## 2. Four connected objectives

In [ ]:
objectives = pd.DataFrame([
    {
        "objective": 1,
        "question": "Are there reproducible and interpretable multi-domain risk profiles?",
        "features": "6 worry + 7 experience features",
        "target": "None; unsupervised learning",
        "task": "Clustering",
        "depends_on": "None",
    },
    {
        "objective": 2,
        "question": "Who is more likely to belong to each frozen profile?",
        "features": "Demographic, socioeconomic, discrimination, support and country-context variables",
        "target": "Frozen profile label from Objective 1",
        "task": "Multiclass classification",
        "depends_on": "Objective 1 completed and frozen",
    },
    {
        "objective": 3,
        "question": "Do profiles add predictive value for an external practical outcome?",
        "features": "Nested context, summary-score, profile-label and individual-risk blocks",
        "target": "External resilience or preparedness outcome",
        "task": "Supervised prediction with nested models",
        "depends_on": "Objective 1 completed and frozen",
    },
    {
        "objective": 4,
        "question": "Does harm in other domains improve prediction of worry in a focal domain?",
        "features": "Context, matching harm, other-domain harm, discrimination and support",
        "target": "Each worry domain, analysed separately",
        "task": "Ordinal classification; nonlinear comparison model",
        "depends_on": "Week 3 prepared variables; conceptually informed by Objective 1",
    },
])
display(objectives)
assert len(objectives) == 4

## 3. Data split and leakage boundary

In [ ]:
country_split = pd.read_csv(ROOT / "processed" / "2023_country_split.csv")
split_summary = country_split.groupby("split").agg(
    countries=("COUNTRY_ISO3", "nunique"),
    respondents=("respondents", "sum"),
)
display(split_summary)

assert country_split["COUNTRY_ISO3"].is_unique
assert set(country_split["split"]) == {"train", "heldout_country"}
assert split_summary.loc["train", "countries"] == 113
assert split_summary.loc["heldout_country", "countries"] == 29

The split holds out entire countries. A random respondent split would place respondents from the same national context in both development and validation data, making generalization look easier than it is.

**Leakage rule:** imputation values, scaling parameters, encoders, model hyperparameters, cluster number and cluster centers are learned using discovery countries only. Held-out countries are used once the workflow is frozen.

## 4. Objective-specific split logic

In [ ]:
split_plan = pd.DataFrame([
    [1, "Country-grouped development/held-out split", "Discover profiles in development countries; evaluate geographic transport in unseen countries"],
    [2, "Country-grouped split; stratify within development data when tuning", "Prevent memorization of national profile prevalence"],
    [3, "Same country groups and same eligible rows across nested models", "Attribute performance differences to predictor blocks, not samples"],
    [4, "Country-grouped split; outcome balance checked within folds", "Test cross-domain prediction in unseen societies"],
], columns=["objective", "split", "reason"])
display(split_plan)

## 5. Variables that must not leak into profile discovery

In [ ]:
excluded_from_clustering = pd.DataFrame({
    "variable family": [
        "Demographics and socioeconomic background",
        "Country identity and context",
        "Discrimination and social or institutional support",
        "Financial resilience and preparedness outcomes",
    ],
    "reason": [
        "Used later to characterize or predict frozen profiles",
        "Profiles should be defined by risk configurations rather than geography",
        "Reserved for explanatory and predictive objectives",
        "External outcomes must not help construct the profiles whose value they test",
    ],
})
display(excluded_from_clustering)

## 6. Week 2 quiz answers

In [ ]:
week02_quiz_answers = pd.DataFrame({
    "quiz item": [
        "W02-Q01 Number of connected objectives",
        "W02-Q02 External validation unit",
        "W02-Q03 Where preprocessing is learned",
        "W02-Q04 Reuse of frozen profile label",
        "W02-Q05 Why unrestricted country identity is excluded",
    ],
    "answer": [
        len(objectives),
        "Entire countries",
        "Discovery or training countries only",
        "Target in Objective 2; predictor tested in Objective 3",
        "It could memorize national profile prevalence rather than generalizable relationships",
    ],
})
display(week02_quiz_answers.style.hide(axis="index"))

## Week 2 conclusion

The project contains four connected objectives. Objective 1 creates and freezes the profile definition. Objectives 2 and 3 reuse that frozen output without changing it. Objective 4 asks a complementary domain-level prediction question. All supervised comparisons use country-grouped validation, and the held-out countries remain outside model development.